> (중요) 여러분 구글 드라이브에 최소 7GB 이상은 확보되어 있어야 합니다!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import unicodedata  # 0번 섹션에 추가 필요

# 📌 경로 설정 (제공해주신 경로 반영)
def normalize_path(path):
    # 1. unicodedata.normalize('NFC', path): 경로 문자열을 NFC 방식으로 통일
    # 2. .strip(): 앞뒤에 붙은 불필요한 공백 제거
    return unicodedata.normalize('NFC', path).strip()

> (주의) 아래 코드는 처음 딱 한 번만!

In [ ]:
import zipfile
import os
import shutil
import time

# 1. 경로 설정
dataset_zip = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/초급_프로젝트_수강생_배포용.zip")
extract_path = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/dataset/")

os.makedirs(extract_path, exist_ok=True)

# 2. 메인 압축파일 해제
print(f"📦 메인 데이터셋 해제 중: {os.path.basename(dataset_zip)}")
with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# 메인 압축파일 삭제 (요청사항)
os.remove(dataset_zip)
print("🗑️ 메인 압축파일 삭제 완료.")

# 3. 내부 이미지 압축파일 통합 해제 로직
print("\n🚀 이미지 폴더 통합 및 내부 압축 해제 시작...")
for file in os.listdir(extract_path):
    if file.endswith(".zip"):
        file_path = os.path.join(extract_path, file)
        
        # 이름 기반 대상 폴더 결정 (train/test 통합)
        if 'train' in file.lower():
            target_folder_name = "train_images"
        elif 'test' in file.lower():
            target_folder_name = "test_images"
        else:
            target_folder_name = file.replace(".zip", "")

        target_subfolder = os.path.join(extract_path, target_folder_name)
        os.makedirs(target_subfolder, exist_ok=True)

        print(f"📂 {file} -> {target_folder_name} 통합 중...")
        
        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            for member in zip_ref.infolist():
                if not member.is_dir():
                    # 내부 경로 구조를 무시하고 파일명만 추출하여 저장
                    filename = os.path.basename(member.filename)
                    if filename:
                        target_file_path = os.path.join(target_subfolder, filename)
                        with zip_ref.open(member) as source, open(target_file_path, "wb") as target:
                            shutil.copyfileobj(source, target)
        
        # [수정 포인트] 삭제 전 잠시 대기 후 강제 삭제 시도
        try:
            time.sleep(0.5) 
            if os.path.exists(file_path):
                os.remove(file_path)
                print(f"🗑️ 삭제 성공: {file}")
        except Exception as e:
            print(f"❌ {file} 삭제 실패: {e}")

print("\n✨ 모든 작업이 완료되었습니다!")
print(f"📁 최종 데이터셋 구성: {os.listdir(extract_path)}")

- 구글 드라이브 휴지통 비우기

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build

# 1. 구글 드라이브 인증
auth.authenticate_user()
drive_service = build('drive', 'v3')

# 2. 휴지통 완전히 비우기 함수
def empty_trash():
    try:
        drive_service.files().emptyTrash().execute()
        print("✅ 구글 드라이브 휴지통이 완전히 비워졌습니다.")
    except Exception as e:
        print(f"❌ 휴지통 비우기 실패: {e}")

# 실행
empty_trash()

> 압축 해제한 파일들의 반영 시간이 걸릴 수 있으므로, 커널 재시작 해주기!

#### `normalize_path`는?

`normalize_path`는 파일 경로에 포함된 **한글(유니코드) 처리 방식**을 통일하여, 경로를 찾지 못하는 에러를 방지하기 위한 함수입니다.

특히 **Google Colab**이나 **Mac, Windows** 사이에서 데이터를 주고받을 때 한글 폴더명이 깨져서 발생하는 `File Not Found` 에러를 잡는 데 필수적입니다.

<br>

##### 왜 사용하나요? (NFC vs NFD)

한글을 컴퓨터가 인식하는 방식은 크게 두 가지입니다.

* **NFC (Windows 스타일):** '강'을 '강'이라는 하나의 글자로 저장합니다.
* **NFD (Mac/Unix 스타일):** '강'을 'ㄱ', 'ㅏ', 'ㅇ'으로 쪼개서 저장합니다.

사람 눈에는 똑같이 "초급 프로젝트"라고 보이지만, 컴퓨터 입장에서는 글자 조합 방식이 다르면 **완전히 다른 경로**로 인식합니다. `normalize_path` 는 이를 **NFC(표준 방식)** 로 강제 통일해주는 역할을 합니다.

In [ ]:
############################################################
# 0. 라이브러리 임포트 & 경로 설정
############################################################
import os
import json
import shutil
import yaml
import random
import unicodedata
import torch
from tqdm import tqdm
from pathlib import Path
from ultralytics import YOLO

# 📌 경로 정규화 함수
def normalize_path(path):
    return unicodedata.normalize('NFC', path).strip()

# 📌 기본 경로 설정
extract_path = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/dataset/")
TRAIN_JSON_PATH = os.path.join(extract_path, "merged_annotations_train_final.json")
TRAIN_IMG_DIR = os.path.join(extract_path, "train_images")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 📌 YOLO 프로젝트 루트 설정
yolo_root = "/content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_format"
train_images_path = os.path.join(yolo_root, "train/images")
train_labels_path = os.path.join(yolo_root, "train/labels")
val_images_path = os.path.join(yolo_root, "val/images")
val_labels_path = os.path.join(yolo_root, "val/labels")

for d in [train_images_path, train_labels_path, val_images_path, val_labels_path]:
    os.makedirs(d, exist_ok=True)

# ---------------------------------------------------------
# 📌 1. 데이터 로드 및 클래스 재매핑
# ---------------------------------------------------------
with open(TRAIN_JSON_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)

id_to_img = {img['id']: img for img in data['images']}
unique_cats = sorted(list(set(ann['category_id'] for ann in data['annotations'])))
cat_to_yolo = {old_id: new_id for new_id, old_id in enumerate(unique_cats)}
class_names = [f"pill_{i}" for i in range(len(unique_cats))]

# ---------------------------------------------------------
# 📌 2. YOLO 포맷 변환 및 파일 복사
# ---------------------------------------------------------
for ann in tqdm(data['annotations']):
    img_info = id_to_img.get(ann['image_id'])
    if not img_info: continue
    
    file_name = img_info['file_name']
    w_orig, h_orig = img_info['width'], img_info['height']
    
    # 좌표 변환 (Center_x, Center_y, Width, Height)
    x, y, w, h = ann['bbox']
    x_center = (x + w / 2) / w_orig
    y_center = (y + h / 2) / h_orig
    norm_w = w / w_orig
    norm_h = h / h_orig
    
    # 레이블 저장 (기존 내용 덮어쓰기 방지를 위해 'a' 모드로 열되, 이미지당 하나씩 생성되도록 관리)
    txt_name = os.path.splitext(file_name)[0] + ".txt"
    label_file = os.path.join(train_labels_path, txt_name)
    
    with open(label_file, 'a') as f:
        new_class_id = cat_to_yolo[ann['category_id']]
        f.write(f"{new_class_id} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}\n")

    # 이미지 파일 복사 (한 번만 복사)
    src_img = os.path.join(TRAIN_IMG_DIR, file_name)
    dst_img = os.path.join(train_images_path, file_name)
    if not os.path.exists(dst_img):
        shutil.copy2(src_img, dst_img)

# ---------------------------------------------------------
# 📌 3. 검증 데이터셋(Val) 분리 (9:1)
# ---------------------------------------------------------
all_images = [f for f in os.listdir(train_images_path) if f.endswith(('.png', '.jpg', '.jpeg'))]
val_count = int(len(all_images) * 0.1)
val_files = random.sample(all_images, val_count)

for f_name in val_files:
    shutil.move(os.path.join(train_images_path, f_name), os.path.join(val_images_path, f_name))
    t_name = os.path.splitext(f_name)[0] + ".txt"
    if os.path.exists(os.path.join(train_labels_path, t_name)):
        shutil.move(os.path.join(train_labels_path, t_name), os.path.join(val_labels_path, t_name))

# ---------------------------------------------------------
# 📌 4. data.yaml 생성 및 학습 시작
# ---------------------------------------------------------
yaml_data = {
    'path': yolo_root,
    'train': 'train/images',
    'val': 'val/images',  # 📌 분리된 val 폴더를 정확히 지칭
    'nc': len(class_names),
    'names': class_names
}

yaml_save_path = os.path.join(yolo_root, 'data.yaml')
with open(yaml_save_path, 'w', encoding='utf-8') as f:
    yaml.dump(yaml_data, f, allow_unicode=True)

print(f"✅ 설정 완료! 학습을 시작합니다.")

model = YOLO('yolov8n.pt')
model.train(
    data=yaml_save_path,
    epochs=20,
    imgsz=640,
    batch=16,
    device=DEVICE,
    

    # [비교 모델 faster-R-CNN과 같은 조건으로 데이터 증강 설정] 
    mosaic=0.0,       # ❌ Faster R-CNN에는 없는 기능이므로 끔 (중요!)
    mixup=0.0,        # ❌ 이것도 끔
    copy_paste=0.0,   # ❌ 이것도 끔
    
    fliplr=0.5,       # ✅ 좌우 반전 50% (T.functional.hflip과 동일)
    flipud=0.5,       # ✅ 상하 반전 50% (T.functional.vflip과 동일)
    
    # ✅ ColorJitter와 유사한 색감 변화 설정
    hsv_h=0.1,        # Hue (색상) 변화량
    hsv_s=0.4,        # Saturation (채도) 변화량
    hsv_v=0.4,        # Brightness (명도) 변화량
    
    # 그 외 부가적인 기법들도 비교를 위해 0으로 만듭니다.
    degrees=0.0,      # 회전 끔
    translate=0.0,    # 이동 끔
    scale=0.0,        # 크기 조절 끔
    shear=0.0,        # 비틀기 끔
    perspective=0.0   # 투영 변환 끔
    
)